In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()
os.environ['GOOGLE_API_KEY'] = os.getenv('GEMINI-API')
os.environ['GROQ_API_KEY'] = os.getenv('GROQ-API')

# model = init_chat_model("groq:qwen/qwen3.6-27b")
model = init_chat_model("google_genai:gemini-3.5-flash-lite")

## Structured Ouput
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output.

## Pydantic
Pydantic models provide the richest feature set with field validations, descriptions and nested structures.

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    director: str = Field(description="The director of the movie")
    rating:float = Field(description="The movies rating out of 10")

    

In [4]:
model_with_structure = model.with_structured_output(Movie)

model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 3.5 Flash Lite', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.5-flash-lite', temperature=None, client=<google.genai.client.Client object at 0x7fef73096990>, default_met

In [6]:
model.invoke("Provide the details about the movie : Avengers : End Game")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content=[{'type': 'text', 'text': '**Avengers: Endgame** is a landmark 2019 superhero film produced by Marvel Studios and distributed by Walt Disney Studios Motion Pictures. It is the 22nd film in the Marvel Cinematic Universe (MCU) and serves as the direct sequel to *Avengers: Infinity War* (2018), acting as the epic culmination of the 22-film story arc known as the "Infinity Saga."\n\nHere are the comprehensive details about the movie:\n\n### 1. Production & Crew\n* **Directors:** Anthony and Joe Russo (The Russo Brothers)\n* **Writers:** Christopher Markus and Stephen McFeely (based on the Marvel Comics characters)\n* **Producers:** Kevin Feige\n* **Music Composer:** Alan Silvestri\n* **Cinematography:** Trent Opaloch\n* **Budget:** ~$356 million (making it one of the most expensive films ever made)\n* **Release Date:** April 26, 2019 (United States)\n\n### 2. Main Cast & Characters\n* **Robert Downey Jr.** as Tony Stark / Iron Man\n* **Chris Evans** as Steve Rogers / Capt

In [7]:
model_with_structure.invoke("Provide the details about the movie : Avengers : End Game")

Movie(title='Avengers: Endgame', year=2019, director='Anthony and Joe Russo', rating=8.4)

## Message output alongside parsed structure

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    director: str = Field(description="The director of the movie")
    rating:float = Field(description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie avengers: Endgame")
response

{'raw': AIMessage(content=[{'type': 'text', 'text': '{\n  "title": "Avengers: Endgame",\n  "year": 2019,\n  "director": "Anthony and Joe Russo",\n  "rating": 8.4\n}', 'extras': {'signature': 'El4KXAERTTIPkm45+3l2hL8f1+MlysoW/YicPpRoLLUW2mMP4xmoPqzdnSzI7VGmlmjysfSc1TxQA3ke0ZKCEWE2crZnbeW7tcEv4nhbt2ZAypT7ZnF7S7zo2uDR61H/'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0a05a-3ec4-77b0-9769-2b9176d106a4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 44, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}}),
 'parsed': Movie(title='Avengers: Endgame', year=2019, director='Anthony and Joe Russo', rating=8.4),
 'parsing_error': None}

## Nested Structure

In [13]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Avengers Endgame")
response

MovieDetails(title='Avengers: Endgame', year=2019, cast=[Actor(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), Actor(name='Chris Evans', role='Steve Rogers / Captain America'), Actor(name='Mark Ruffalo', role='Bruce Banner / Hulk'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Scarlett Johansson', role='Natasha Romanoff / Black Widow')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=356.0)

## Typed Dict

Typed dict provides simpler alternative using Python's built in typing, ideal when you dont need run time validation

In [14]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypeddict = model.with_structured_output(MovieDict)
model_withtypeddict.invoke("Please provide the details of movie Avengers End game")

{'title': 'Avengers: Endgame',
 'year': 2019,
 'director': 'Anthony and Joe Russo',
 'rating': 8.4}

In [16]:
model.profile

{'name': 'Gemini 3.5 Flash Lite',
 'release_date': '2026-07-21',
 'last_updated': '2026-07-21',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True,
 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'],
 'reasoning_effort_default': 'minimal'}

### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using @dataclass decorator.

In [19]:
from traitlets.utils.descriptions import describe
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description = "The email address of the person")
    phone: str = Field(description = "The phone no of the person")

model = ChatGoogleGenerativeAI(model = 'gemini-3.6-flash')

agent = create_agent(
    model = model,
    response_format = ContactInfo
)

result = agent.invoke({
    "messages" : [{"role" : "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='ccfc863a-9392-4923-9528-85cec13859de'),
  AIMessage(content=[{'type': 'text', 'text': '{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', 'extras': {'signature': 'EukJCuYJARFNMg9WDLBkfL+mc3lsoNkdIOax5OFtc9IIt8d9ausPPYsN1iWrHVjFvXMuJisDoKOLSC2MXZPgweJOaVWfBy8IETxsXFILqeJatYDjRHOq4yFy6Ao2Q7/fF88dsyvVbGK0TZ4/lFhuPrtjlEVZqzcTBVxAWs8Afy1T2+557w3CNhvWT1+kF/mit/CY+xaSLPu87HfDhu7sxPEbe8fb/dq3T/swWA8moNX87+Zrz90Cpr0eS6cvw0WqXK/bULZMqlL/VmXH6qFJei8KPDZoorTUsJILP4sdt8tH6nf5mkZzVdy1ukTHOWo3C2LDVdNPmZCgMXfOk76yF3TzqFpOmcfYCBJuWJnzTTkRE30mKYkxWLUPq8FEA6I0T4N3uiUgYn5PLDEJT2tpQPtD1vAaY5PHeM9OdFETxpvXuo7nKQ9fk6YNDH3IiP3HTu0FqVKMQRKkvE4OkRHGRXk0/HwTnuXRUrjOyT5G7Li3KHJTzihz8hVxtBELxMlBhSGH12oRzrSfW1QK65/JIDEgNUbe10/Qdjtt5hQRjS01vjxFCEKK2JfNmzio5n8q9zgM+48cS8erJSEBHPGAGSGaXQmsFk/dAQv8I1fy72Y/ew/2vQhbbylOR21A71khRIrwkkJprEaY5bMmey

In [ ]:
## Dataclass
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

@dataclass
class ContactInfo:
    name: str  # the name of the person
    email: str # the email address of the person
    phone: str # the phone number of the person

model = ChatGoogleGenerativeAI(model = 'gemini-3.6-flash')

agent = create_agent(
    model = model,
    response_format = ContactInfo
)

result = agent.invoke({
    "messages" : [{"role" : "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='763e6a13-050d-40d8-a5ef-dd7d779d1b76'),
  AIMessage(content=[{'type': 'text', 'text': '{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', 'extras': {'signature': 'EvUJCvIJARFNMg/aMywZFNjB1btg6/M0Ra29qhyAxfPIInYw/ZxM0+sT9AJe1wtBEebbxOGgjrsK3nZsWcJZdcHzVgh8S5GBYWNulvkqUIhmpKP5B59bqa8p3DRf8R51Qdh7l2/kCYZZhuZpANWFvTbtfWqfyY0Opx/UNoxB5d3H5OgqZgKb2i31dXBIwauIgeU9cMHWDem/FNUepTkdXSTSrs/BLGA0iZx0B8MKHVX6m2AR/OFbLSMe8xLNGPSjC5xynDbY2lh735EQQb3+4om9LeuFNeHCuOB6+qfTLzHuX5r+Fw9ny4o/1cDDh7l5pJhuQ6gQFgmouBF1lrUWimJjOCGl+zEBB8umU0H1xQ/H8Fu2hiGyj9zK+wCp4wQzBORKvpkmRp8pxUssofOfCNzZaqjUId44BpMLlfZOQjJi61Tn64fKFlMHDmO+AqJEbeLPJOI6svTtkXkb8pqPJV9f+8K1a4R5N98us72z+zpc1nJiKG3OSTq+mUpUirTZU7AuRa9VRQRb8hTFzdVK2q+TlZ1fKHbQp3CP28wOtqC2mbi9XOf4ATp5ntnZFy+aA+X73W/bcbCsZL5RVmLpcN3ARO2jNLc1A8wcA30fbM3F/lUEctuiQYoZyKnNlMNzAeWOjVf9pEpp+/AIzs